# Validation vs OOT Feature Drift — Memory-Safe

Designed for the large Freddie Mac behavioral datasets. The notebook minimizes repeated full scans and avoids collecting large Spark objects to the driver.

**Reference:** validation. **Comparison:** OOT. OOT is not used for calibration.


In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
from pathlib import Path
import pandas as pd
import numpy as np
project_path = Path.cwd().parent
# ---- SET THESE ----
VAL_PATH = Path("data/04_model_split/behavioral/validation_split.parquet")  # set to validation parquet path/directory
OOT_PATH = Path("data/04_model_split/behavioral/oot_split.parquet")  # set to OOT parquet path/directory

OUTPUT_DIR = Path('./drift_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: use the exact model feature list here if available.
# Leave None to automatically use common numeric/string/bool columns.
MODEL_FEATURES = None

ID_COLUMNS = {
    'loan_id', 'monthly_reporting_period', 'observation_date',
    'label', 'target', 'y', 'default_flag', 'event_flag'
}
PREDICTION_COLUMNS = {
    'prediction', 'predicted_pd', 'pd', 'score', 'raw_prediction',
    'probability', 'prediction_pd'
}

spark = (SparkSession.builder
    .appName('mortgage-credit-risk-feature-drift')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '512')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/vorad/mortgage-credit-risk/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/04 15:45:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/04 15:45:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 4.2.0


In [2]:
assert VAL_PATH is not None, 'Set VAL_PATH first.'
assert OOT_PATH is not None, 'Set OOT_PATH first.'

val_df = spark.read.parquet(f"{project_path}/{VAL_PATH}")
oot_df = spark.read.parquet(f"{project_path}/{OOT_PATH}")



val_n = val_df.count()
oot_n = oot_df.count()
print(f'Validation rows: {val_n:,}')
print(f'OOT rows:        {oot_n:,}')
print(f'Validation partitions: {val_df.rdd.getNumPartitions()}')
print(f'OOT partitions:        {oot_df.rdd.getNumPartitions()}')


Validation rows: 15,818,929
OOT rows:        6,138,195
Validation partitions: 32
OOT partitions:        32


In [3]:
# Common schema / feature discovery
val_cols, oot_cols = set(val_df.columns), set(oot_df.columns)
common_cols = sorted(val_cols & oot_cols)
print('Common columns:', len(common_cols))
print('Validation-only:', sorted(val_cols - oot_cols))
print('OOT-only:', sorted(oot_cols - val_cols))

NUMERIC_TYPES = (T.ByteType,T.ShortType,T.IntegerType,T.LongType,T.FloatType,T.DoubleType,T.DecimalType)
val_fields = {f.name:f for f in val_df.schema.fields}
        
if MODEL_FEATURES is None:
    numeric_features = []
    categorical_features = []
    for c in common_cols:
        if c in ID_COLUMNS or c.lower() in PREDICTION_COLUMNS:
            continue
        f = val_fields[c]
        if isinstance(f.dataType, NUMERIC_TYPES):
            numeric_features.append(c)
        elif isinstance(f.dataType, (T.StringType,T.BooleanType)):
            categorical_features.append(c)
else:
    MODEL_FEATURES = [c for c in MODEL_FEATURES if c in common_cols]
    numeric_features = [c for c in MODEL_FEATURES if isinstance(val_fields[c].dataType, NUMERIC_TYPES)]
    categorical_features = [c for c in MODEL_FEATURES if isinstance(val_fields[c].dataType, (T.StringType,T.BooleanType))]

print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)


Common columns: 92
Validation-only: []
OOT-only: []
Numeric features: 74
Categorical features: 17
Numeric: ['borrower_assistance_count_12m', 'borrower_assistance_count_3m', 'calculated_loan_age', 'credit_score', 'current_actual_upb', 'current_delinquency_flag', 'current_delinquency_streak', 'current_dpd_30_plus', 'current_dpd_60_plus', 'current_dpd_numeric', 'current_interest_bearing_upb', 'current_interest_rate', 'current_non_interest_bearing_upb', 'ddlpi', 'ddlpi_ahead_flag', 'delinquency_episode_count', 'delinquency_intensity_change_6m', 'delinquency_months_12m', 'delinquency_months_3m', 'delinquency_months_to_date', 'delinquent_accrued_interest', 'disaster_delinquency_count_12m', 'disaster_delinquency_count_3m', 'dpd_30_count_12m', 'dpd_30_count_3m', 'dpd_60_count_12m', 'dpd_60_count_3m', 'dpd_acceleration_6m', 'dpd_severity_change_6m', 'dpd_trend_6m', 'estimated_ltv', 'ever_30dpd_to_date', 'ever_60dpd_to_date', 'ever_borrower_assistance', 'ever_disaster_delinquency', 'ever_modifie

## 1. One-pass numerical summary

Missingness, mean, standard deviation, min and max are calculated for all numerical features in one aggregation per population. This avoids the previous per-feature `count()` explosion.


In [4]:
c

26/09/04 15:45:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/04 15:45:17 ERROR Executor: Exception in task 1.0 in stage 8.0 (TID 69)32]
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.internal.config.ConfigReader.<init>(ConfigReader.scala:54)
	at org.apache.spark.sql.internal.ReadOnlySQLConf.<init>(ReadOnlySQLConf.scala:37)
	at org.apache.spark.sql.internal.SQLConf$.get(SQLConf.scala:223)
	at org.apache.spark.sql.catalyst.rules.RuleExecutor.execute(RuleExecutor.scala:222)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateMutableProjection$.$anonfun$canonicalize$1(GenerateMutableProjection.scala:36)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateMutableProjection$$$Lambda/0x000000a002161198.apply(Unknown Source)
	at scala.collection.immutable.ArraySeq.map(ArraySeq.scala:75)
	at scala.collection.immutable.ArraySe

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

## 2. Quantile drift for important variables

Approximate quantiles are intentionally limited to the variables most relevant to the GAM diagnosis. This is much cheaper than calculating them for every column.


In [5]:
IMPORTANT_FEATURES = [
    'calculated_loan_age','estimated_ltv','credit_score','original_ltv',
    'original_dti','original_cltv','current_interest_rate','current_actual_upb',
    'dpd_trend_6m','dpd_acceleration_6m','dpd_severity_change_6m',
    'upb_pct_change_from_origination','rate_change_from_origination'
]
IMPORTANT_FEATURES = [c for c in IMPORTANT_FEATURES if c in numeric_features]
QUANTILES = [.01,.05,.10,.25,.50,.75,.90,.95,.99]

q_rows=[]
for c in IMPORTANT_FEATURES:
    vq = val_df.approxQuantile(c, QUANTILES, .005)
    oq = oot_df.approxQuantile(c, QUANTILES, .005)
    q_rows += [{'feature':c,'quantile':q,'validation':v,'oot':o,'shift':o-v}
               for q,v,o in zip(QUANTILES,vq,oq)]

quantile_drift = pd.DataFrame(q_rows)
display(quantile_drift)
quantile_drift.to_csv(OUTPUT_DIR/'important_quantile_drift.csv',index=False)


ConnectionRefusedError: [Errno 111] Connection refused

## 3. Numerical PSI — controlled pass

PSI bins are defined from validation deciles and then applied to both populations. We deliberately do this only after the cheap summary stage, and one feature at a time to keep JVM memory bounded.


In [ ]:
PSI_EPS = 1e-6

def calc_psi_feature(c):
    qs=[i/10 for i in range(1,10)]
    knots=sorted(set(float(x) for x in val_df.approxQuantile(c,qs,.005) if x is not None and np.isfinite(float(x))))
    if len(knots)<2:
        return {'feature':c,'psi':0.0,'n_bins':1}

    # Validation-derived edges. Spark bucket assignment happens without collecting rows.
    edges=[-float('inf')]+knots+[float('inf')]
    x=F.col(c).cast('double')
    expr=F.lit(len(edges)-2)
    for i in range(len(edges)-2,-1,-1):
        expr=F.when(x<=F.lit(edges[i+1]),F.lit(i)).otherwise(expr)

    vcounts=(val_df.select(expr.alias('_bin')).where(F.col('_bin').isNotNull())
             .groupBy('_bin').count().collect())
    ocounts=(oot_df.select(expr.alias('_bin')).where(F.col('_bin').isNotNull())
             .groupBy('_bin').count().collect())
    vm={int(r['_bin']):int(r['count']) for r in vcounts}
    om={int(r['_bin']):int(r['count']) for r in ocounts}
    n=len(edges)-1
    vp=np.clip(np.array([vm.get(i,0) for i in range(n)],float)/val_n,PSI_EPS,None)
    op=np.clip(np.array([om.get(i,0) for i in range(n)],float)/oot_n,PSI_EPS,None)
    value=float(np.sum((op-vp)*np.log(op/vp)))
    return {'feature':c,'psi':value,'n_bins':n}

psi_rows=[]
for i,c in enumerate(numeric_features,1):
    print(f'[{i}/{len(numeric_features)}] {c}')
    psi_rows.append(calc_psi_feature(c))

numeric_psi=pd.DataFrame(psi_rows).sort_values('psi',ascending=False)
display(numeric_psi)
numeric_psi.to_csv(OUTPUT_DIR/'numeric_psi.csv',index=False)


## 4. Categorical drift

Categoricals are aggregated one feature at a time. This is intentional: a single pathological high-cardinality field should not consume the entire JVM heap.


In [ ]:
def js_divergence(a,b):
    p=np.asarray(a,float); q=np.asarray(b,float)
    p=p/max(p.sum(),1); q=q/max(q.sum(),1)
    p=np.clip(p,1e-12,None); q=np.clip(q,1e-12,None); m=(p+q)/2
    return float(.5*np.sum(p*np.log(p/m))+.5*np.sum(q*np.log(q/m)))

def categorical_row(c):
    v=(val_df.select(F.coalesce(F.col(c).cast('string'),F.lit('__NULL__')).alias('_x'))
       .groupBy('_x').count().collect())
    o=(oot_df.select(F.coalesce(F.col(c).cast('string'),F.lit('__NULL__')).alias('_x'))
       .groupBy('_x').count().collect())
    vm={str(r['_x']):int(r['count']) for r in v}; om={str(r['_x']):int(r['count']) for r in o}
    cats=sorted(set(vm)|set(om))
    vp=[vm.get(x,0) for x in cats]; op=[om.get(x,0) for x in cats]
    return {'feature':c,'psi':float(np.sum((np.clip(np.array(op,float)/oot_n,PSI_EPS,None)-np.clip(np.array(vp,float)/val_n,PSI_EPS,None))*
        np.log(np.clip(np.array(op,float)/oot_n,PSI_EPS,None)/np.clip(np.array(vp,float)/val_n,PSI_EPS,None)))),
        'js_divergence':js_divergence(vp,op),'validation_unique':len(vm),'oot_unique':len(om),
        'new_categories':sum(x not in vm for x in cats),'disappeared_categories':sum(x not in om for x in cats),
        'validation_missing_pct':100*vm.get('__NULL__',0)/val_n,'oot_missing_pct':100*om.get('__NULL__',0)/oot_n}

cat_rows=[]
for i,c in enumerate(categorical_features,1):
    print(f'[{i}/{len(categorical_features)}] {c}')
    cat_rows.append(categorical_row(c))

categorical_drift=pd.DataFrame(cat_rows).sort_values('psi',ascending=False)
display(categorical_drift)
categorical_drift.to_csv(OUTPUT_DIR/'categorical_drift.csv',index=False)


In [ ]:
# Final ranking
combined=pd.concat([
    numeric_psi[['feature','psi']].assign(type='numeric'),
    categorical_drift[['feature','psi']].assign(type='categorical')
],ignore_index=True).sort_values('psi',ascending=False)
combined['drift_band']=pd.cut(combined['psi'],[-np.inf,.05,.10,.25,np.inf],labels=['minimal','minor','moderate','major'])
display(combined)
combined.to_csv(OUTPUT_DIR/'feature_drift_ranking.csv',index=False)

print('\nPSI guide: <0.05 minimal | 0.05-0.10 minor | 0.10-0.25 moderate | >=0.25 major')
print(f'Results: {OUTPUT_DIR.resolve()}')
